# 06 · Experiment E1: which input channels? (task T22, runs R01 to R09)

**RQ1.** Does extra radar information help? Three inputs are compared, everything else being the frozen default (MobileNetV2 encoder, BCE + Dice loss):

| Config | Input channels | Runs |
|---|---|---|
| VH only | VH | R04, R05, R06 |
| VV+VH | VV, VH | R07, R08, R09 |
| Default | VV, VH and the ratio VV − VH | R01, R02, R03 |

Three seeds per config show how much a result moves by chance alone. A difference is only claimed when it is larger than that.

The training defaults were frozen and tagged in git (`defaults-frozen`, task T19) **before** this notebook was first run. The first cell checks that the code still matches that tag.

In [ ]:
import json
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "config.py").exists())
sys.path.insert(0, str(ROOT))
import src  # noqa: F401  (must come before TensorFlow: switches on the XLA compile cache)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src import analysis, viz
from src.config import STUDIES, load_config, run_table, runs_of
from src.runner import check_frozen, run_benchmark, run_evaluation, run_many

viz.use_style()
pd.set_option("display.float_format", lambda v: f"{v:.4f}")
print("training and scoring code is identical to the frozen tag 'defaults-frozen' (commit %s)" % check_frozen()[:7])

STUDY = "E1"
CONFIGS = STUDIES[STUDY]["configs"]
pd.DataFrame([r for r in run_table() if r["config_name"] in CONFIGS]).fillna({"channels": "vv_vh_ratio"}).set_index("run_id")

## 1. R01 on test and Bolivia: scored once, now that the defaults are frozen

R01 was trained and inspected on validation data in notebook 05. Its test and Bolivia scores were deliberately not computed until the method was fixed.

In [ ]:
_ = run_evaluation("R01")

## 2. Train and score R02 to R09

Each run trains in its own process, then is scored once on test and Bolivia. Finished runs are not repeated when the notebook is run again; their saved logs are shown.

In [ ]:
results = run_many(["R02", "R03", "R04", "R05", "R06", "R07", "R08", "R09"])

## 3. Results

One row per run. Validation IoU is the score the checkpoint was chosen by; test and Bolivia were scored once, after training.

In [ ]:
runs = analysis.runs_table()
study_runs = runs[runs["Config"].isin(CONFIGS)]
study_runs[["Run ID", "Config", "Seed", "Status", "Epochs trained", "Best epoch", "Val IoU (best)", "Test IoU",
            "Test precision", "Test recall", "Bolivia IoU", "Sec per epoch"]].set_index("Run ID")

Mean ± standard deviation over the three seeds (the two threshold baselines, single deterministic runs, are shown for reference):

In [ ]:
analysis.study_table(STUDY, analysis.summary_table(runs))

**Is a difference real?** The project's rule: a gap between two configs counts only if it is larger than the seed-to-seed standard deviation.

In [ ]:
gaps = analysis.gap_table(STUDY, runs)
gaps.to_csv(ROOT / "results" / f"{STUDY}_gaps.csv", index=False)
gaps.set_index(["Metric", "A", "B"])

In [ ]:
hist = {c: analysis.histories(runs_of(c)) for c in CONFIGS}
axes = viz.plot_val_curves(hist)
axes[0].figure.tight_layout()
viz.savefig(axes[0].figure, ROOT / "figures" / f"{STUDY}_val_curves.png")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.3), sharey=True)
done = runs[runs["Status"] == "Finished"]
for ax, metric in zip(axes, ("Test IoU", "Bolivia IoU")):
    viz.plot_config_bars(done, CONFIGS, metric, ax, baseline=("Otsu (E0)", "Tuned VH threshold"))
    ax.set_title(metric.replace("IoU", "water IoU"))
fig.tight_layout()
viz.savefig(fig, ROOT / "figures" / f"{STUDY}_bars.png")
plt.show()